# LangGraph 入門介紹

## 什麼是 LangGraph?

LangGraph 是一個**低階的 AI 代理編排框架**。

### 用白話文解釋:
- **代理 (Agent)**: 就像一個會自己思考、做決定的 AI 助手
- **編排 (Orchestration)**: 就是安排這些 AI 助手按照一定的流程工作
- **低階框架**: 表示它給你很大的自由度,讓你可以細緻地控制每一個步驟

LangGraph 可以與 LangChain 整合使用,但也可以單獨運作。

## 我們要做什麼?

我們將建立一個**圖 (Graph)**,這個圖可以整合不同的模型和工具。可以把它想像成一個流程圖,AI 會按照這個流程圖來工作。

## 六個步驟

本教學基於 [LangGraph 快速入門指南](https://docs.langchain.com/oss/python/langgraph/quickstart),分為六個步驟:

1. **定義工具和模型** - 選擇要用哪個 AI 模型,以及提供什麼工具給它
2. **定義狀態** - 決定要記錄什麼資訊 (像是對話記錄)
3. **定義模型節點** - 設定 AI 如何思考和回應
4. **定義工具節點** - 設定工具如何被使用
5. **定義結束邏輯** - 決定什麼時候該繼續,什麼時候該停止
6. **建立並編譯代理** - 把所有東西組合起來,讓它可以運作

# 步驟 1: 定義工具和模型

## 我們要用什麼?

- **AI 模型**: OpenAI 的 GPT-4o-mini (一個聰明的語言模型)
- **工具**: 兩個可以查詢動物知識的公開 API
  - [Meowfacts API](https://github.com/wh-iterabb-it/meowfacts) - 提供貓咪的有趣知識
  - [Dog Facts API](https://kinduff.github.io/dog-api/) - 提供狗狗的有趣知識

## 為什麼需要工具?

AI 模型本身只能根據它訓練時學到的知識來回答。但如果我們給它「工具」,它就可以:
- 查詢最新的資訊
- 執行特定的功能
- 獲取即時數據

就像給一個學生一本字典,他就能查到更多資訊!

## 初始化聊天模型

首先,我們要初始化一個 [聊天模型](https://docs.langchain.com/oss/python/langchain/models#initialize-a-model)。

### 程式碼解釋:
- `init_chat_model()`: 初始化一個聊天模型
- `"openai:gpt-4o-mini"`: 指定使用 OpenAI 的 GPT-4o-mini 模型
- `temperature=0.7`: 控制回答的創意程度
  - 0.0 = 非常保守、可預測的回答
  - 1.0 = 非常有創意、隨機的回答
  - 0.7 = 平衡創意與穩定性

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0.7
)

## 測試模型

模型可以直接使用,但我們稍後會把它整合到一個更大的「圖」結構中。

讓我們先簡單測試一下模型能不能正常回答問題:

In [ ]:
# 測試:問模型一個問題
response = model.invoke("為什麼鸚鵡會說話?")
print(response)

In [ ]:
# 查看完整的回應資訊 (JSON 格式)
response.to_json()

## 定義工具

使用 LangChain 來定義工具。注意每個函數前面的 `@tool` 裝飾器。

### 什麼是裝飾器 (@tool)?

裝飾器就像是給函數加上一個「標籤」,告訴 LangChain:「這個函數是一個工具,AI 可以使用它!」

### 程式碼重點解釋:

1. **函數說明文件 (docstring)**:
   - 三引號內的文字會被 AI 讀取
   - AI 會根據這些說明來決定什麼時候使用這個工具

2. **API 呼叫流程**:
   - 發送 HTTP 請求到 API
   - 接收 JSON 格式的回應
   - 解析資料並格式化成易讀的文字

3. **`bind_tools(tools)`**:
   - 把工具「綁定」到模型上
   - 這樣模型就知道它可以使用這些工具了

In [ ]:
from langchain.tools import tool
import requests
import json

@tool
def get_cat_facts(n:int=1):
    """
    從 Meowfacts API 獲取 n 個貓咪小知識。
    當使用者想知道關於貓的有趣事實時,使用這個工具。
    """
    url = "https://meowfacts.herokuapp.com/"
    params = {
        "count": n  # 要獲取幾個知識
    }
    # 發送 GET 請求到 API
    response = requests.get(url, params=params)
    # 解析 JSON 回應
    resp_dict = json.loads(response.text)
    facts_list = resp_dict.get("data", [])
    # 格式化成易讀的文字
    facts = "\n".join([f"{i+1}. {fact}\n" for i, fact in enumerate(facts_list)])
    return facts

@tool
def get_dog_facts(n:int=1):
    """
    從 Dog API 獲取 n 個狗狗小知識。
    當使用者想知道關於狗的有趣事實時,使用這個工具。
    """
    url = "http://dogapi.dog/api/v2/facts"
    params = {
        "limit": n  # 要獲取幾個知識
    }
    response = requests.get(url, params=params)
    resp_dict = json.loads(response.text)
    facts_list = resp_dict.get("data", [])
    # 從 API 回應中提取實際的知識內容
    facts = "\n".join([f"{i+1}. {fact['attributes']['body']}\n" for i, fact in enumerate(facts_list)])
    return facts


# 建立工具列表
tools = [get_cat_facts, get_dog_facts]
# 建立工具字典,方便之後用名稱查找工具
tools_by_name = {tool.name: tool for tool in tools}
# 把工具綁定到模型上,讓 AI 知道它可以使用這些工具
model_with_tools = model.bind_tools(tools)

# 步驟 2: 定義狀態

## 什麼是「狀態」(State)?

狀態就像是 AI 的「記事本」,用來記錄:
- **對話歷史** - AI 和使用者說了什麼
- **AI 呼叫次數** - AI 思考了幾次

## 技術細節

這是 `langgraph.MessagesState` 的擴展版本,我們額外加入了「LLM 呼叫次數」的記錄。

### 程式碼解釋:

- **`TypedDict`**: Python 的型別提示,幫助我們定義資料結構
- **`Annotated[list[AnyMessage], operator.add]`**: 
  - `Annotated`: 加上額外的說明
  - `operator.add`: 表示新訊息要「加入」到列表中,而不是「取代」原有的
  - 這樣可以保留完整的對話歷史

- **`llm_calls: int`**: 記錄 AI 被呼叫了幾次

In [ ]:
from langchain_core.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    """
    定義圖的狀態結構
    """
    # 訊息列表:儲存所有的對話內容
    # operator.add 確保新訊息會被加入列表,而不是取代整個列表
    messages: Annotated[list[AnyMessage], operator.add]
    
    # LLM 呼叫次數:記錄 AI 被呼叫了幾次
    llm_calls: int

# 步驟 3: 定義模型節點

## 什麼是「模型節點」?

模型節點是圖中的一個「思考站」,AI 在這裡:
1. 讀取使用者的問題和對話歷史
2. 決定要直接回答,還是需要使用工具
3. 產生回應或工具呼叫

## 流程說明

```
使用者問題 → 模型節點 → 決定:
                          ├─ 直接回答 (結束)
                          └─ 呼叫工具 (前往工具節點)
```

## 程式碼解釋:

- **`SystemMessage`**: 系統訊息,給 AI 設定「角色」和「任務」
  - 就像告訴員工「你的工作是...」
  
- **回傳值**:
  - `messages`: 新增 AI 的回應到對話歷史
  - `llm_calls`: 呼叫次數 +1

In [ ]:
from langchain_core.messages import SystemMessage


def llm_call(state: dict):
    """
    LLM 節點:AI 在這裡決定要直接回答還是呼叫工具
    """
    return {
        "messages": [
            model_with_tools.invoke(
                [
                    # 系統訊息:設定 AI 的角色
                    SystemMessage(
                        content="你是一個樂於助人的助手,專門分享關於貓和狗的有趣知識。"
                    )
                ]
                # 加上所有的對話歷史
                + state["messages"]
            )
        ],
        # 呼叫次數 +1
        "llm_calls": state.get('llm_calls', 0) + 1
    }

# 步驟 4: 定義工具節點

## 什麼是「工具節點」?

工具節點是圖中的「執行站」,在這裡:
1. 接收 AI 決定要使用的工具和參數
2. 實際執行工具 (例如:呼叫 API 獲取貓咪知識)
3. 把結果傳回給 AI

## 流程說明

```
AI 說:「我要用 get_cat_facts(n=3)」
         ↓
工具節點:執行 get_cat_facts(n=3)
         ↓
回傳:「1. 貓有9條命... 2. 貓可以跳... 3. 貓會...」
         ↓
回到模型節點,讓 AI 整理成最終回答
```

## 程式碼解釋:

- **`state["messages"][-1].tool_calls`**: 
  - 取得最後一則訊息 (AI 的回應)
  - 從中提取 AI 想要呼叫的工具
  
- **`tool.invoke(tool_call["args"])`**:
  - 用 AI 提供的參數來執行工具
  
- **`ToolMessage`**:
  - 把工具的執行結果包裝成訊息格式
  - 這樣 AI 就能讀取這些結果

In [ ]:
from langchain_core.messages import ToolMessage


def tool_node(state: dict):
    """
    工具節點:執行 AI 要求的工具
    """
    result = []
    
    # AI 可能一次要呼叫多個工具,所以要用迴圈處理
    for tool_call in state["messages"][-1].tool_calls:
        # 根據工具名稱找到對應的工具
        tool = tools_by_name[tool_call["name"]]
        
        # 執行工具,傳入 AI 指定的參數
        observation = tool.invoke(tool_call["args"])
        
        # 把執行結果包裝成 ToolMessage
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    
    return {"messages": result}

# 步驟 5: 定義結束邏輯

## 什麼是「結束邏輯」?

結束邏輯就像是交通警察,決定接下來要:
- **繼續**: 前往工具節點 (因為 AI 需要使用工具)
- **停止**: 結束流程 (因為 AI 已經準備好最終回答)

## 決策流程

```
檢查 AI 的最後一則訊息
         |
    是否包含工具呼叫?
         |
    ┌────┴────┐
   是         否
    |          |
前往工具節點  結束流程
(執行工具)   (回答使用者)
```

## 程式碼解釋:

- **`Literal["tool_node", END]`**: 
  - 型別提示,表示這個函數只會回傳兩種值之一
  - "tool_node" 或 END
  
- **`last_message.tool_calls`**:
  - 檢查 AI 是否想要呼叫工具
  - 如果有,就回傳 "tool_node"
  - 如果沒有,就回傳 END (結束)

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """
    決定是否要繼續執行(前往工具節點)或停止(結束流程)
    
    這是基於 AI 是否做出了工具呼叫來判斷
    """
    messages = state["messages"]
    last_message = messages[-1]  # 取得最後一則訊息

    # 如果 AI 想要呼叫工具,就前往工具節點
    if last_message.tool_calls:
        return "tool_node"

    # 否則,AI 已經準備好最終回答,結束流程
    return END

# 步驟 6: 建立並編譯代理

## 什麼是「圖」(Graph)?

圖就像是一張地圖,標示了:
- **節點 (Nodes)**: 各個工作站 (模型節點、工具節點)
- **邊 (Edges)**: 連接路徑,決定流程如何進行

## 完整流程圖

```
START (開始)
  |
  ↓
llm_call (AI 思考)
  |
  ↓
should_continue (決策)
  |
  ├─→ tool_node (執行工具) ─→ 回到 llm_call
  |
  └─→ END (結束,回答使用者)
```

## 建立步驟說明:

1. **建立圖結構**:
   - `StateGraph(MessagesState)`: 建立一個使用我們定義的狀態結構的圖

2. **新增節點**:
   - `add_node("llm_call", llm_call)`: 新增 AI 思考節點
   - `add_node("tool_node", tool_node)`: 新增工具執行節點

3. **新增邊 (連接路徑)**:
   - `add_edge(START, "llm_call")`: 從開始到 AI 思考節點
   - `add_conditional_edges()`: 從 AI 思考節點根據條件決定下一步
   - `add_edge("tool_node", "llm_call")`: 從工具節點回到 AI 思考節點

4. **編譯**:
   - `compile()`: 把圖編譯成可執行的代理

5. **視覺化**:
   - 顯示圖的結構,讓我們可以看到整個流程

In [ ]:
# 建立工作流程
agent_builder = StateGraph(MessagesState)

# 新增節點 (工作站)
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# 新增邊 (連接路徑)
# 1. 從開始點到 AI 思考節點
agent_builder.add_edge(START, "llm_call")

# 2. 從 AI 思考節點根據條件決定下一步
agent_builder.add_conditional_edges(
    "llm_call",           # 從哪個節點
    should_continue,      # 用什麼函數決定
    ["tool_node", END]    # 可能的目的地
)

# 3. 從工具節點回到 AI 思考節點 (形成循環)
agent_builder.add_edge("tool_node", "llm_call")

# 編譯代理
agent = agent_builder.compile()

# 顯示圖的結構
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# 執行代理

## 如何使用?

我們可以這樣來執行代理:

1. **建立訊息列表**: 包含使用者的問題
2. **呼叫代理**: 使用 `agent.invoke()`
3. **查看結果**: 印出所有的訊息

## 實際執行流程:

```
使用者: "告訴我 3 件關於貓的事"
  ↓
AI 思考: "我需要呼叫 get_cat_facts(n=3)"
  ↓
執行工具: 獲取 3 個貓咪知識
  ↓
AI 整理: 把知識整理成易讀的回答
  ↓
回答使用者
```

然後繼續處理第二個問題...

## 程式碼說明:

- **`HumanMessage`**: 代表使用者的訊息
- **`agent.invoke()`**: 執行代理
- **`pretty_print()`**: 以易讀的格式印出訊息

In [ ]:
# 執行代理
from langchain_core.messages import HumanMessage

# 建立訊息列表
messages = [
    HumanMessage(content="告訴我 3 件關於貓的事。"),
    HumanMessage(content="現在告訴我 2 件關於狗的事。")
]

# 呼叫代理
messages = agent.invoke({"messages": messages})

# 印出所有訊息
for m in messages["messages"]:
    m.pretty_print()

# 總結

## 我們學到了什麼?

1. **LangGraph 的基本概念**:
   - 圖 (Graph): 一個工作流程的藍圖
   - 節點 (Nodes): 各個工作站
   - 邊 (Edges): 連接路徑
   - 狀態 (State): 記憶系統

2. **如何建立一個 AI 代理**:
   - 定義 AI 模型
   - 提供工具給 AI 使用
   - 設計工作流程
   - 編譯並執行

3. **工作流程的運作**:
   - AI 接收問題
   - 決定是否需要工具
   - 執行工具獲取資訊
   - 整理並回答使用者

## 延伸學習

現在你已經了解基礎了,可以嘗試:

1. **新增更多工具**: 例如查詢天氣、搜尋網頁等
2. **調整 AI 的行為**: 修改系統訊息,讓 AI 有不同的個性
3. **建立更複雜的流程**: 加入更多節點和決策點
4. **加入錯誤處理**: 當工具失敗時該怎麼辦

## 重要概念複習

| 概念 | 說明 | 比喻 |
|------|------|------|
| Agent (代理) | 會自己思考和行動的 AI | 聰明的助手 |
| Tool (工具) | AI 可以使用的功能 | 助手的工具箱 |
| Node (節點) | 流程中的一個步驟 | 工作站 |
| Edge (邊) | 連接節點的路徑 | 道路 |
| State (狀態) | 記錄的資訊 | 記事本 |
| Graph (圖) | 整個工作流程 | 地圖 |

## 實用建議

- **從簡單開始**: 先建立基本的代理,再逐步增加功能
- **測試每個部分**: 確保每個節點和工具都能正常運作
- **視覺化流程**: 使用圖來檢查流程是否符合預期
- **記錄狀態**: 適當地記錄資訊,方便除錯

祝你學習愉快! 🚀